# Eval-awareness probes on a Colab A100


## 1. a100 confirmation


In [ ]:
import subprocess, sys, os
print('python :', sys.executable)
print('cwd    :', os.getcwd())
print('colab  :', os.path.exists('/content'))
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
with open('/proc/meminfo') as f:
    print('Total RAM:', round(int(f.readline().split()[1]) / 1e6, 1), 'GB')

## 2. Get the repo onto the runtime


In [ ]:

REPO = 'https://github.com/ianshxn/eval-awareness-probe-proto.git'

!rm -rf /content/eap
!git clone -q $REPO /content/eap
!git -C /content/eap log --oneline -1

%cd /content/eap
!ls

## 3. Dependencies

`transformers==4.49.0` is pinned for TransformerLens compatibility and downgrades the
preinstalled version, so the kernel must restart (notebook toolbar -> **Restart**).

The VM's disk survives that restart -- the clone from step 2 is still there -- but the
Python process doesn't, so `%cd` and all variables reset. After restarting, re-run
step 4 onward; it re-establishes the working directory on its own.

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed. Restart the kernel now, then continue at step 4.')

## 4. Re-establish state after the restart

In [ ]:
%cd /content/eap
MODEL = 'microsoft/Phi-3-medium-4k-instruct'
# Fallback if the medium load fails: 'microsoft/Phi-3-mini-4k-instruct'

import transformers, torch
print('transformers', transformers.__version__, '(expect 4.49.0)')
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 5. Generate vectors (README step 3)

204 contrastive examples x 2 forward passes. The activation hook prints per layer per
example, so output is very chatty -- VS Code truncates long streams, which is fine since
the vectors go to disk. `--layers` is capped for a first run; drop it to sweep all 40.

In [ ]:
!python scripts/generate_vectors.py \
  --model $MODEL \
  --data datasets/contrastive_dataset.json \
  --output phi3medium_contrastive \
  --layers 10,15,20,25,30,35

In [ ]:
import glob
run_dir = sorted(glob.glob('phi3medium_contrastive_*'))[-1]
print(run_dir)
!cat {run_dir}/experiment_info.txt
!ls {run_dir}/normalized_vectors

## 7. Build the SAD stages_oversight eval set, then analyze (README step 4)

This is the paper's **probe-selection benchmark**. The vectors in step 6 were trained on
`contrastive_dataset.json`, so scoring them against SAD `stages_oversight` is genuinely
out-of-distribution -- the same train-on-contrastive / select-on-SAD setup the paper uses
to rank probes by AUROC.

`analyze_probe.py` labels each example with `item.get('type') == 'testing'` and applies
the chat template itself, so the eval file just needs a list of
`{"question": <raw text>, "type": "testing" | "deployment"}`. None of the four shipped
datasets carry a `type` field, so run as-is they collapse to a single class and the AUC is
undefined. The cell below builds a correctly-labelled SAD file instead.

**Where SAD lives.** The oversight split ships inside a password-protected zip in the
`LRudL/sad` repo (`sad/stages/private_data_gen.zip`, password `sadtimesforthesetimes`,
members `out/test_oversight.jsonl` and `out/deploy_oversight.jsonl`). Each row has a `body`
(the text) and `choices_right`, which holds `'yes'` for a testing/oversight item and `'no'`
for a deployment one -- we label by that content, not by which file the row came from, and
balance the two classes so the AUROC is not skewed by base rate.

The build is done **inline** here rather than via a repo script, because the clone in step 2
only brings *your* repo onto the VM. (`scripts/build_sad_oversight_eval.py` in the repo does
the same thing for local runs -- use it if you commit and push it first.)

**Validity check:** pass `--control` to the analyze cell (step 19) to compare against random
probes with matched statistics. On a benchmark you did not train on, that is the number
worth trusting.

In [ ]:
# Build the SAD stages_oversight eval set on the VM (self-contained -- no repo push needed).
import os, json, zipfile, random

if not os.path.exists('/content/sad'):
    !git clone -q https://github.com/LRudL/sad.git /content/sad

ZIP = '/content/sad/sad/stages/private_data_gen.zip'
PW  = b'sadtimesforthesetimes'

def load_jsonl(name):
    with zipfile.ZipFile(ZIP) as z:
        text = z.read(name, pwd=PW).decode('utf-8')
    return [json.loads(line) for line in text.strip().split('\n')]

items = load_jsonl('out/deploy_oversight.jsonl') + load_jsonl('out/test_oversight.jsonl')

# choices_right holds 'yes' for a testing/oversight item and 'no' for deployment --
# label by that content, not by which file the row came from.
testing    = [{'question': it['body'], 'type': 'testing'}    for it in items if 'yes' in it['choices_right']]
deployment = [{'question': it['body'], 'type': 'deployment'} for it in items if 'no'  in it['choices_right']]

rng = random.Random(0)
rng.shuffle(testing); rng.shuffle(deployment)
n = min(len(testing), len(deployment))          # balance the classes for a clean AUROC
labelled = testing[:n] + deployment[:n]
rng.shuffle(labelled)

os.makedirs('datasets', exist_ok=True)
with open('datasets/sad_oversight_labelled.json', 'w', encoding='utf-8') as f:
    json.dump(labelled, f, indent=2, ensure_ascii=True)
print(f'Wrote {len(labelled)} labelled examples ({n} testing / {n} deployment) '
      f'to datasets/sad_oversight_labelled.json')

`analyze_dataset_with_probe` runs a **fresh forward pass per example per layer**, so the
build cell's printed count x 6 layers is the total forward passes through a 14B model. The
balanced SAD oversight set is a few hundred examples per class -- add `--limit` to the build
cell above (e.g. `--limit 200`, or slice `labelled` before writing) for a faster first pass.

In [ ]:
!python scripts/analyze_probe.py \
  --model $MODEL \
  --vectors {run_dir}/normalized_vectors \
  --data datasets/sad_oversight_labelled.json \
  --output {run_dir}/analysis \
  --visualize

`process_directory` prints AUROC per layer above. Note the polarity convention in
`calculate_auroc_from_csv`: `y_true` is 1 for **deployment**, so a probe that points at
the testing direction shows up as AUROC well *below* 0.5 rather than above it -- read
`abs(auroc - 0.5)` as the strength and the side as the sign.

Add `--control` to the cell above to compare against random probes with matched
statistics. That is the check worth running before believing any single number.

In [ ]:
# Plots render inline over the kernel protocol, so these do come back to VS Code.
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{run_dir}/analysis/visualizations/*.png')):
    print(p); display(Image(p))

## 8. Get results off the VM

The remote kernel cannot write to your local disk, and `drive.mount()` is unreliable
here -- so nothing above is saved anywhere durable yet.

The vectors are small (5120 floats per layer), so pushing them to a private HF dataset
repo is the least painful programmatic route. Manual alternative: zip the run directory
and pull it down through the extension's Colab *Files* pane.

In [ ]:
!zip -qr {run_dir}.zip {run_dir}
!ls -lh {run_dir}.zip   # then download via the Files pane

# Or push to a private HF repo:
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo('YOUR_USER/eap-vectors', repo_type='dataset',
#                 private=True, exist_ok=True)
# api.upload_folder(folder_path=run_dir, repo_id='YOUR_USER/eap-vectors',
#                   repo_type='dataset')